In [1]:
! pip install transformers[torch] datasets scikit-learn accelerate sentencepiece captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.9 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.2

In [2]:
import os
import json
import glob
import re
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import multiprocessing
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from captum.attr import LayerIntegratedGradients

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
def clean_text_advanced(text):
    if not text: return ""
    text = text.replace('·', ' * ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_question(q_text):
    if not q_text: return ""
    q_text = re.sub(r'^\d+[\s\.\-\,]+', '', q_text)
    return clean_text_advanced(q_text)

def load_and_flatten_jsonl(folder_path):
    all_questions, all_essays, all_labels = [], [], []
    search_pattern = os.path.join(folder_path, "*_essay_evaluated.jsonl")
    jsonl_files = glob.glob(search_pattern)
    print(f"Found {len(jsonl_files)} JSONL files.")
    
    for file_path in jsonl_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line: continue
                try:
                    data = json.loads(line)
                    answer_variants = data.get("answer_variants", {})
                    variant_evaluations = data.get("variant_evaluations", {})
                    
                    for variant_name, evaluation_data in variant_evaluations.items():
                        essay_text = clean_text_advanced(answer_variants.get(variant_name, ""))
                        if not essay_text: continue
                            
                        evaluations = evaluation_data.get("evaluations", [])
                        for eval_item in evaluations:
                            question = clean_question(eval_item.get("question", ""))
                            satisfied = eval_item.get("satisfied", None)
                            if question and satisfied is not None:
                                all_questions.append(question)
                                all_essays.append(essay_text)
                                all_labels.append(1 if satisfied else 0)
                except Exception as e:
                    print(f"Error parsing line {line_num} in {os.path.basename(file_path)}: {e}")
                
    return pd.DataFrame({"question": all_questions, "essay_text": all_essays, "label": all_labels})

In [5]:
DATA_FOLDER_PATH = "/kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions"  
df = load_and_flatten_jsonl(DATA_FOLDER_PATH)

if df.empty:
    raise ValueError("No data found in the specified path!")

df = df.drop_duplicates(subset=['question', 'essay_text'], keep='first').reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['label'])

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})

num_cores = multiprocessing.cpu_count()

class AntiOverfittingTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(label_smoothing=0.1)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='binary'),
        "precision": precision_score(labels, preds, average='binary'),
        "recall": recall_score(labels, preds, average='binary')
    }

MODELS_TO_TRY = [
    "google/electra-base-discriminator",
    "allenai/scibert_scivocab_uncased",
    "allenai/longformer-base-4096"
]

Found 4 JSONL files.


In [6]:
final_leaderboard = []

for trial_num, model_name in enumerate(MODELS_TO_TRY, 1):
    safe_name = model_name.replace("/", "-")
    output_directory = f"/kaggle/working/outputs-{safe_name}" 
    
    print("\n" + "="*60)
    print(f"STARTING TRIAL #{trial_num}: {model_name}")
    print("="*60)
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        def preprocess_function(examples):
            prompts = [
                f"Instruction: Check if the essay satisfies the criteria: '{q}'.\n\nEssay:\n{e}"
                for q, e in zip(examples['question'], examples['essay_text'])
            ]
            current_max = 1024 if "longformer" in model_name else 512
            return tokenizer(prompts, truncation=True, max_length=current_max)

        print(f"Running Advanced Tokenization for {model_name}...")
        tokenized_datasets = dataset.map(
            preprocess_function, 
            batched=True, 
            remove_columns=['question', 'essay_text'],
            num_proc=num_cores,
            load_from_cache_file=False 
        )
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, 
            num_labels=2,
            hidden_dropout_prob=0.2,       
            attention_probs_dropout_prob=0.2
        )
        
        if "longformer" in model_name:
            model.config.pad_token_id = tokenizer.pad_token_id
            
        model.float() 
        model.to(device)
        
        current_batch = 8 if "longformer" in model_name else 16
        
        training_args = TrainingArguments(
            output_dir=output_directory,       
            learning_rate=2e-5,                        
            per_device_train_batch_size=current_batch,             
            per_device_eval_batch_size=current_batch,
            num_train_epochs=2,                        
            weight_decay=0.02,                
            lr_scheduler_type="cosine",       
            warmup_ratio=0.1,                 
            eval_strategy="steps",               
            eval_steps=1000,                  
            save_strategy="steps",                     
            save_steps=1000,                  
            save_total_limit=2,        
            load_best_model_at_end=True,               
            metric_for_best_model="f1",                
            logging_steps=500,                         
            report_to="none",
            fp16=False,                         
            bf16=False                          
        )
        
        trainer = AntiOverfittingTrainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_datasets['train'],
            eval_dataset=tokenized_datasets['validation'],
            processing_class=tokenizer,  
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )
        
        resume_checkpoint = False
        if os.path.exists(output_directory):
            checkpoints = glob.glob(os.path.join(output_directory, "checkpoint-*"))
            if checkpoints:
                resume_checkpoint = True
                print(f"[Kaggle Environment] Found existing checkpoint in /kaggle/working/. Resuming training automatically!")
        
        print(f"Training {model_name}...")
        trainer.train(resume_from_checkpoint=resume_checkpoint)
        
        history = trainer.state.log_history
        train_steps, train_loss = [], []
        eval_steps, eval_loss, eval_accuracy, eval_f1 = [], [], [], []
        
        print("\n--- Detailed Performance Logs ---")
        for log in history:
            if 'loss' in log and 'eval_loss' not in log:
                train_steps.append(log['step'])
                train_loss.append(log['loss'])
                print(f"🔹 [Step {log['step']:5d}] | Training Loss: {log['loss']:.4f}")
            elif 'eval_loss' in log:
                step = log['step']
                eval_steps.append(step)
                eval_loss.append(log['eval_loss'])
                eval_accuracy.append(log['eval_accuracy'])
                eval_f1.append(log['eval_f1'])
                print(f"\n[Step {step:5d}] ---------- VALIDATION METRICS ----------")
                print(f"    ↳ Validation Loss : {log['eval_loss']:.4f}")
                print(f"    ↳ Accuracy        : {log['eval_accuracy']:.4f}")
                print(f"    ↳ F1-Score        : {log['eval_f1']:.4f}\n" + "-"*50)
        
        plt.figure(figsize=(15, 5))
        plt.subplot(1, 2, 1)
        plt.plot(train_steps, train_loss, label='Training Loss', color='blue')
        if eval_steps: plt.plot(eval_steps, eval_loss, label='Validation Loss', color='red', marker='o')
        plt.title(f'Loss - {model_name}')
        plt.grid(True); plt.legend()

        plt.subplot(1, 2, 2)
        if eval_steps:
            plt.plot(eval_steps, eval_accuracy, label='Accuracy', color='green', marker='s')
            plt.plot(eval_steps, eval_f1, label='F1-Score', color='purple', marker='^')
        plt.title(f'Metrics - {model_name}')
        plt.grid(True); plt.legend()
        
        plot_path = f"/kaggle/working/metrics_plot_{safe_name}.png"
        plt.savefig(plot_path, dpi=300)
        plt.close()
        
        best_f1 = max(eval_f1) if eval_f1 else 0
        best_acc = max(eval_accuracy) if eval_accuracy else 0
        final_leaderboard.append({"Model": model_name, "Best F1-Score": best_f1, "Best Accuracy": best_acc})
        
        save_path = f"/kaggle/working/best_model_{safe_name}"
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Model successfully saved to {save_path}")
        
    except Exception as e:
        print(f"Error occurred in trial {model_name}: {e}")
        import traceback
        traceback.print_exc()
        continue


STARTING TRIAL #1: google/electra-base-discriminator


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Training allenai/longformer-base-4096...


model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Initializing global attention on CLS token...
Initializing global attention on CLS token...
Input ids are automatically padded to be a multiple of `config.attention_window`: 512
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Error occurred in trial allenai/longformer-base-4096: Caught StopIteration in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/longformer/modeling_longformer.py", line 1707, in forward
    outputs = self.longformer(
              ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_ca

Traceback (most recent call last):
  File "/tmp/ipykernel_23/2254531163.py", line 87, in <cell line: 0>
    trainer.train(resume_from_checkpoint=resume_checkpoint)
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2174, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2536, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 3809, in training_step
    loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/4097972121.py", line 23, in compute_loss
    outputs = model(**inputs)
              ^^^^^^^^^^^^^^^
  File "/usr/local/lib/pytho

In [7]:
leaderboard_df = pd.DataFrame(final_leaderboard)
print(leaderboard_df.to_string(index=False))

                            Model  Best F1-Score  Best Accuracy
google/electra-base-discriminator       0.715109       0.744151
 allenai/scibert_scivocab_uncased       0.720123       0.747011
